In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
SELECTED = ['empirical_complete', 'lognormal_grw', 'ar1_s', 'hurdle_ar1_s_p', 'se_hurdle_s']
LABELS = {'empirical_complete':'DBLP empirical', 'lognormal_grw':'GRW', 'ar1_s':'AR(1)',
          'hurdle_ar1_s_p':'Hurdle AR(1)', 'se_hurdle_s':'Self-exciting hurdle'}
palette = sns.color_palette('colorblind')
COLORS = dict(zip(SELECTED, ['#222222', palette[0], palette[1], palette[4], palette[2]]))
LINES = dict(zip(SELECTED, ['-', '--', '-.', ':', '-']))
MARKERS = dict(zip(SELECTED, ['o', 's', '^', 'D', 'v']))
sns.set_theme(style='ticks', context='paper', font_scale=1,
              rc={'font.size':8, 'axes.labelsize':8, 'axes.titlesize':9,
                  'xtick.labelsize':7, 'ytick.labelsize':7, 'legend.fontsize':8,
                  'pdf.fonttype':42, 'ps.fonttype':42, 'savefig.dpi':400})

In [ ]:
qq = pd.read_csv(INPUT/'window_qq_points.csv')
qq = qq.loc[qq.source.isin(SELECTED) & qq.window.eq('career_total')].copy()
qq_summary = pd.read_csv(INPUT/'window_qq_summary.csv')
qq_summary = qq_summary.loc[qq_summary.source.isin(SELECTED) & qq_summary.window.eq('career_total')].copy()
ranks = pd.read_csv(INPUT/'rank_mixing.csv')
ranks = ranks.loc[ranks.source.isin(SELECTED) & ranks.measure.eq('annual') & ranks.reference.eq('lag_5')].copy()
comparison = pd.read_csv(INPUT/'model_comparison.csv').set_index('source').loc[SELECTED]
cdf_knots = pd.read_csv(INPUT/'distribution_cdf.csv')
cdf_knots = cdf_knots.loc[cdf_knots.source.isin(SELECTED) & cdf_knots.diagnostic.eq('career_total')].copy()

In [ ]:
# full observations for exact ECDFs; upstream diagnostics establish the data contract
panel = pd.read_csv(INPUT/'simulation/fit/data/panel_complete.csv', index_col='dblp_id')
totals = {'empirical_complete':panel.to_numpy(dtype=float).sum(axis=1)}
for tag in SELECTED[1:]:
    array = np.load(INPUT/'simulation'/tag/'trajectories.npy', mmap_mode='r')
    totals[tag] = array.sum(axis=0)
total_frame = pd.concat([pd.DataFrame({'source':tag, 'total':values}) for tag, values in totals.items()], ignore_index=True)

In [ ]:
# source-level summaries only; no individual career records exported
summary = comparison[['n', 'mean_career_total', 'gini_career_total', 'final_zero_fraction', 'lag_5_rho20']].copy()
shape = qq_summary.set_index('source')
summary['career_total_qq_r2'] = shape.qq_r2
summary['n_positive_total'] = shape.n_positive
summary['label'] = pd.Series(LABELS)
summary.to_csv(OUTPUT/'pnas_model_summary.csv')
qq.to_csv(OUTPUT/'pnas_model_qq_points.csv', index=False)
ranks.to_csv(OUTPUT/'pnas_model_lag5.csv', index=False)
cdf_knots.to_csv(OUTPUT/'pnas_model_ecdf_knots.csv', index=False)
display(summary)

In [ ]:
def draw_qq(ax):
    for tag in SELECTED:
        data = qq.loc[qq.source.eq(tag)].sort_values('normal_quantile')
        sns.lineplot(data=data, x='normal_quantile', y='observed_quantile', ax=ax,
                     color=COLORS[tag], linestyle=LINES[tag], marker=MARKERS[tag],
                     markevery=50, markersize=2.5, linewidth=1.1, estimator=None, errorbar=None, legend=False)
    limits = (qq.normal_quantile.min(), qq.normal_quantile.max())
    ax.plot(limits, limits, color='.6', linewidth=.7, zorder=0)
    ax.set(title='Career-total shape', xlabel='Standard normal quantile',
           ylabel='Standardized log-total quantile')
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    sns.despine(ax=ax)
def draw_ecdf(ax):
    for tag in SELECTED:
        sns.ecdfplot(data=total_frame.loc[total_frame.source.eq(tag)], x='total', ax=ax,
                     color=COLORS[tag], linestyle=LINES[tag], linewidth=1.2, legend=False)
    # shared scale and full tails; linear segment preserves the mass at zero
    ax.set_xscale('symlog', linthresh=1, linscale=.5, base=10)
    ax.set_xlim(left=0)
    ax.set(title='Career-total scale', xlabel='Career-total output', ylabel='Cumulative probability',
           ylim=(0, 1.02), yticks=[0, .25, .5, .75, 1])
    ax.text(.98, .05, 'Log scale above 1', transform=ax.transAxes, ha='right', fontsize=6.5)
    sns.despine(ax=ax)
def draw_rank(ax):
    for tag in SELECTED:
        data = ranks.loc[ranks.source.eq(tag)].sort_values('age')
        sns.lineplot(data=data, x='age', y='rho', ax=ax, color=COLORS[tag],
                     linestyle=LINES[tag], marker=MARKERS[tag], markevery=3,
                     markersize=3, linewidth=1.2, estimator=None, errorbar=None, legend=False)
    ax.set(title='Annual rank memory', xlabel='Career year, $t$',
           ylabel=r'Spearman $\rho(R_t,R_{t-5})$',
           xlim=(ranks.age.min(), ranks.age.max()), ylim=(-.05, 1.05))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    sns.despine(ax=ax)
def legend_handles():
    return [Line2D([], [], color=COLORS[tag], linestyle=LINES[tag], marker=MARKERS[tag],
                   markersize=3, linewidth=1.2, label=LABELS[tag]) for tag in SELECTED]
def save_figure(fig, stem):
    for extension in ['pdf', 'png']:
        fig.savefig(OUTPUT/f'{stem}.{extension}', bbox_inches='tight', facecolor='white')

In [ ]:
# two-column manuscript composite; shared legend above the panels
fig, axes = plt.subplots(1, 3, figsize=(7.1, 3.25))
for letter, draw, ax in zip('ABC', [draw_qq, draw_ecdf, draw_rank], axes):
    draw(ax)
    ax.text(-.18, 1.09, letter, transform=ax.transAxes, weight='bold', fontsize=10)
fig.legend(handles=legend_handles(), loc='upper center', ncol=3, frameon=False,
           bbox_to_anchor=(.5, 1.01), columnspacing=1.4, handlelength=2.8)
fig.subplots_adjust(left=.075, right=.99, bottom=.19, top=.73, wspace=.55)
save_figure(fig, 'pnas_dblp_model_comparison')
plt.show()
plt.close(fig)

In [ ]:
# individual panels for alternative manuscript layouts
for name, draw in [('qq', draw_qq), ('ecdf', draw_ecdf), ('lag5', draw_rank)]:
    fig, ax = plt.subplots(figsize=(3.5, 3.3))
    draw(ax)
    fig.legend(handles=legend_handles(), loc='upper center', ncol=2, frameon=False,
               bbox_to_anchor=(.5, 1.01), handlelength=2.8)
    fig.subplots_adjust(left=.20, right=.96, bottom=.17, top=.70)
    save_figure(fig, f'pnas_dblp_{name}')
    plt.close(fig)

In [ ]:
# caption records transformations, cohorts and interpretation limits
n_empirical = int(comparison.loc['empirical_complete', 'n'])
n_simulated = int(comparison.loc['lognormal_grw', 'n'])
last_year = int(ranks.age.max())
caption = (
    r'\textbf{Distributional shape and temporal agreement of DBLP productivity models.} '
    f'Empirical comparisons use {n_empirical:,} complete publication histories over career years '
    f'0--{last_year}; each model generates {n_simulated:,} histories. '
    r'(A) Normal QQ curves of positive career totals after taking $\log_2$ and standardizing '
    r'each source separately to zero mean and unit sample standard deviation. '
    r'The grey diagonal is the normal reference. Standardization compares shape while removing '
    r'differences in location and scale; zero totals are excluded only from this panel. '
    r'QQ curves retain up to 401 ordered points, including the extremes. '
    r'(B) Exact empirical cumulative distribution functions of career totals, including zero totals '
    r'and the full tails. All sources share an output axis that is linear from zero to one '
    r'and logarithmic above one. '
    r'(C) Spearman correlation of annual productivity ranks five years apart, with average ranks '
    r'for zero ties. The same complete empirical cohort is used at every year. '
    r'Models share colors and line styles across panels. These are in-sample generative checks. '
    r'The self-exciting hurdle model is fitted to complete histories; the other models use '
    r'available adjacent-year observations, so this is not a controlled same-cohort ablation.'
)
(OUTPUT/'pnas_dblp_model_comparison_caption.tex').write_text(caption+'\n')
display(pd.DataFrame({'file':['pnas_dblp_model_comparison.pdf', 'pnas_dblp_model_comparison.png',
                             'pnas_dblp_model_comparison_caption.tex', 'pnas_model_summary.csv'],
                      'purpose':['three-panel manuscript figure', 'preview',
                                 'LaTeX caption', 'source-level numeric summary']}))